# Illustrated Training And Optimizers

Training has a loop:

```txt
data -> model -> prediction -> loss
                         |
                         v
parameters <- optimizer <- gradients
```

The loss measures wrongness. Gradients describe how parameters affect that wrongness. The optimizer changes parameters.

When reading this notebook, keep asking: what value is stored, what value is consumed, and what value changes?


In [ ]:
import { tensor, parameter, matmul, add, mean, grad, clear_grad, clip_grad_norm, no_grad, copy, sgd, adam, adamw, schedules, Duration, scheduled } from 'affon:compute/legacy'

function show(label: string, value: { shape: readonly number[], to_array(): unknown } | null) {
  console.log(label)
  if (value === null) {
    console.log('null')
    return
  }
  console.log('shape:', value.shape)
  console.log(JSON.stringify(value.to_array()))
}


## Parameters And Gradients

`parameter` creates trainable state. `grad(loss, params)` computes gradients and stores them on parameters.

$$
\theta \leftarrow \theta - \eta \nabla_\theta L
$$

Read this as: parameter moves opposite the loss slope by a learning-rate-sized step.

<svg viewBox="0 0 760 230" width="760" height="230" xmlns="http://www.w3.org/2000/svg">
  <style>.txt{font:14px sans-serif;fill:#0f172a}.mono{font:15px monospace}.box{fill:#dbeafe;stroke:#60a5fa}.loss{fill:#fee2e2;stroke:#f87171}.grad{fill:#ffedd5;stroke:#fb923c}.arrow{stroke:#475569;stroke-width:2;marker-end:url(#arrow)}</style>
  <defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#475569"/></marker></defs>
  <rect class="box" x="80" y="75" width="110" height="50" rx="4"/><text class="mono" x="112" y="105">data</text>
  <line class="arrow" x1="190" y1="100" x2="270" y2="100"/>
  <rect class="box" x="270" y="75" width="120" height="50" rx="4"/><text class="mono" x="305" y="105">model</text>
  <line class="arrow" x1="390" y1="100" x2="470" y2="100"/>
  <rect class="loss" x="470" y="75" width="110" height="50" rx="4"/><text class="mono" x="505" y="105">loss</text>
  <line class="arrow" x1="525" y1="125" x2="525" y2="170"/>
  <rect class="grad" x="455" y="170" width="140" height="42" rx="4"/><text class="mono" x="488" y="196">gradients</text>
  <line class="arrow" x1="455" y1="191" x2="355" y2="191"/>
  <rect class="box" x="230" y="170" width="125" height="42" rx="4"/><text class="mono" x="255" y="196">parameters</text>
  <text class="txt" x="80" y="36">Optimizer reads gradients and updates parameters.</text>
</svg>


### Before You Run

Predict which objects will get a `.grad`: the input data, the prediction, or the parameters?

### Read The Output

Gradients are attached to `W` and `b` because those are the trainable parameters passed to `grad(loss, [W, b])`.


In [ ]:
const W = parameter([2, 1]).ones()
const b = parameter([1]).zeros()
const x = tensor([[1, 2], [3, 4]])

clear_grad([W, b])
const pred = add(matmul(x, W), b)
const loss = mean(pred)
grad(loss, [W, b])

show('prediction', pred)
show('loss', loss)
show('W.grad', W.grad)
show('b.grad', b.grad)


## Clearing And Clipping Gradients

Gradients are stored on parameters. `clear_grad` removes old gradients before the next batch.

`clip_grad_norm` rescales gradients when their global norm is too large.


### Before You Run

After `clear_grad([W, b])`, predict what `W.grad` should be.

### Debugging Cue

If training behaves strangely, check whether gradients are accumulating from previous batches.


In [ ]:
console.log('grad norm before clipping:', clip_grad_norm([W, b], 1.0))
show('W.grad after clipping', W.grad)
clear_grad([W, b])
show('W.grad after clear_grad', W.grad)


## SGD, Adam, AdamW

- `sgd`: move opposite the gradient.
- `adam`: keep running averages of gradients and squared gradients.
- `adamw`: Adam plus decoupled weight decay.

Start with SGD to learn the mechanics. Use Adam as a strong default when training a real model.


### Before You Run

For a positive gradient and learning rate `0.1`, predict whether SGD moves a parameter up or down.


In [ ]:
const wSgd = parameter([1]).ones()
clear_grad([wSgd])
grad(mean(wSgd), [wSgd])
const sgdStep = sgd({ lr: 0.1 })
sgdStep([wSgd])
show('w after SGD step', wSgd)

const adamStep = adam({ lr: 0.01 })
const adamwStep = adamw({ lr: 0.01, weight_decay: 0.01 })
console.log('adam lr:', adamStep.lr)
console.log('adamw lr:', adamwStep.lr)


## Schedules

A schedule changes the learning rate over steps or epochs.

<svg viewBox="0 0 760 220" width="760" height="220" xmlns="http://www.w3.org/2000/svg">
  <style>.axis{stroke:#475569;stroke-width:2}.grid{stroke:#e2e8f0}.linear{fill:none;stroke:#2563eb;stroke-width:4}.cos{fill:none;stroke:#16a34a;stroke-width:4}.step{fill:none;stroke:#f97316;stroke-width:4}.txt{font:14px sans-serif;fill:#0f172a}.mono{font:15px monospace}</style>
  <text class="txt" x="24" y="28">Learning-rate schedules decide step size over time.</text>
  <line class="axis" x1="80" y1="170" x2="700" y2="170"/><line class="axis" x1="80" y1="170" x2="80" y2="50"/>
  <path class="linear" d="M100 160 L260 70"/><text class="mono" x="115" y="64">linear</text>
  <path class="cos" d="M310 70 C360 70 410 155 470 160"/><text class="mono" x="350" y="64">cosine</text>
  <path class="step" d="M520 75 L590 75 L590 120 L660 120 L660 155 L700 155"/><text class="mono" x="535" y="64">step</text>
  <text class="txt" x="690" y="190">step</text><text class="txt" x="42" y="58">lr</text>
</svg>


### Before You Run

The linear schedule goes from `0` to `1` over two steps. Predict values at steps `0`, `1`, and `2`.


In [ ]:
const schedule = schedules.sequence(
  schedules.linear({ start: 0, end: 1, duration: Duration.steps(2) }),
  schedules.constant(1),
)
console.log('schedule step 0:', schedule({ step: 0, epoch: 0 }))
console.log('schedule step 1:', schedule({ step: 1, epoch: 0 }))
console.log('schedule step 2:', schedule({ step: 2, epoch: 0 }))

const scheduledStep = scheduled(sgd({ lr: 0.1 }), schedule)
scheduledStep.epoch(0)
console.log('scheduled step lr before use:', scheduledStep.lr)


## no_grad And copy

`no_grad` runs code without recording training graph work. `copy(target, source)` overwrites tensor contents.

Use these for evaluation, manual state updates, and controlled parameter changes.


In [ ]:
const target = parameter([2]).zeros()
no_grad(() => copy(target, tensor([5, 6])))
show('target after no_grad copy', target)
